In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load data
df = pd.read_csv('data/LengthOfStay.csv')

# 2. Drop leakage columns and our previous regression target
df_clean = df.drop(columns=['eid', 'vdate', 'discharged', 'lengthofstay'])
df_model = pd.get_dummies(df_clean, drop_first=True)

# 3. Separate features (X) from our new target (y)
X = df_model.drop(columns=['dialysisrenalendstage'])
y = df_model['dialysisrenalendstage']

# 4. Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# 1. Initialize and train the baseline classifier
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# 2. Get standard predictions (0 or 1) and probabilities (0.0 to 1.0)
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

# 3. Print the evaluation metrics
print("--- Logistic Regression Baseline ---")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.3f}")

--- Logistic Regression Baseline ---
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     19263
           1       0.47      0.11      0.17       737

    accuracy                           0.96     20000
   macro avg       0.72      0.55      0.58     20000
weighted avg       0.95      0.96      0.95     20000

ROC-AUC Score: 0.861


In [3]:
!pip install xgboost


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import xgboost as xgb

# 1. Calculate the exact imbalance ratio (Healthy count / Sick count)
imbalance_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]
print(f"Weighting minority class by: {imbalance_ratio:.2f}x")

# 2. Train the XGBoost model with the penalty weight
xgb_model = xgb.XGBClassifier(scale_pos_weight=imbalance_ratio, random_state=42)
xgb_model.fit(X_train, y_train)

# 3. Predict and evaluate
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\n--- XGBoost Classifier (Balanced) ---")
print(classification_report(y_test, y_pred_xgb))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob_xgb):.3f}")

Weighting minority class by: 26.54x

--- XGBoost Classifier (Balanced) ---
              precision    recall  f1-score   support

           0       0.99      0.94      0.96     19263
           1       0.31      0.75      0.44       737

    accuracy                           0.93     20000
   macro avg       0.65      0.84      0.70     20000
weighted avg       0.96      0.93      0.94     20000

ROC-AUC Score: 0.946


In [5]:
import joblib

joblib.dump(xgb_model, 'xgb_model.joblib')
joblib.dump(list(X_train.columns), 'classification_features.joblib')
print("Classification model saved!")

Classification model saved!


In [6]:
# 1. Separate features for Respiratory Risk (predicting Pneumonia)
X_resp = df_model.drop(columns=['pneum'])
y_resp = df_model['pneum']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_resp, y_resp, test_size=0.2, random_state=42)

# 2. Handle imbalance and train
imbalance_r = y_train_r.value_counts()[0] / y_train_r.value_counts()[1]
resp_model = xgb.XGBClassifier(scale_pos_weight=imbalance_r, random_state=42)
resp_model.fit(X_train_r, y_train_r)

# 3. Save the brain and the checklist for the new model
import joblib
joblib.dump(resp_model, 'resp_model.joblib')
joblib.dump(list(X_train_r.columns), 'resp_features.joblib')
print("Respiratory model trained and saved!")

Respiratory model trained and saved!
